## Setup

In [ ]:
import os
import numpy as np
import pandas as pd

In [ ]:
!cp -r /kaggle/input/datasets/theoviel/biohub-cell-tracking-code/src ./

In [ ]:
try:
    import zarr, geff, tracksdata
    
except: 
    import os
    import sys
    import subprocess
    import importlib
    import importlib.util
    from pathlib import Path
    
    os.environ.setdefault("POLARS_PREFER_PKG", "32")
    
    SUPPORT_DIR = Path(
        "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1"
    )
    WHEELS_DIR = SUPPORT_DIR / "wheels"
    
    if not WHEELS_DIR.exists():
        candidates = list(Path("/kaggle/input").glob("**/wheels"))
        if not candidates:
            raise FileNotFoundError("Could not find the attached offline wheels directory")
        WHEELS_DIR = candidates[0]
    
    print("Offline wheels:", WHEELS_DIR)
    
    OFFLINE_PACKAGES = [
        "tracksdata",
        "zarr==3.2.1",
        "numcodecs==0.15.1",
        "donfig==0.8.1.post1",
        "geff==1.2.0.1.1",
        "geff-spec==1.1.1",
        "pyscipopt==6.2.1",
        "ilpy==0.6.0",
        "rustworkx==0.18.0",
        "polars==1.42.0",
        "polars-runtime-32==1.42.0",
        "bidict==0.23.1",
        "imagecodecs==2026.6.26",
    ]
    
    def module_missing(module_name: str) -> bool:
        return importlib.util.find_spec(module_name) is None
    
    
    REQUIRED_IMPORTS = {
        "tracksdata": "tracksdata",
        "zarr": "zarr",
        "numcodecs": "numcodecs",
        "geff": "geff",
        "pyscipopt": "pyscipopt",
        "ilpy": "ilpy",
        "rustworkx": "rustworkx",
        "polars": "polars",
        "imagecodecs": "imagecodecs",
    }
    
    
    def purge_modules(module_roots):
        """
        Remove already-imported package modules from sys.modules.
    
        Normally this cell runs before imports, but this also protects against
        accidental imports performed by earlier Kaggle initialization code.
        """
        for root in module_roots:
            for name in list(sys.modules):
                if name == root or name.startswith(root + "."):
                    sys.modules.pop(name, None)
    
    
    def install_offline_packages():
        cmd = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-index",
            "--no-deps",
            "--find-links",
            str(WHEELS_DIR),
            *OFFLINE_PACKAGES,
        ]
    
        print("Installing attached packages without modifying NumPy/SciPy/Torch...")
        result = subprocess.run(
            cmd,
            text=True,
            capture_output=True,
        )
    
        if result.returncode != 0:
            print(result.stdout[-4000:])
            print(result.stderr[-4000:])
            raise RuntimeError("Offline dependency installation failed")
    
        purge_modules(REQUIRED_IMPORTS.values())
    
    
    install_offline_packages()
    
    
    failures = {}
    
    for name, module_name in {
        **REQUIRED_IMPORTS,
        "numpy": "numpy",
        "scipy": "scipy",
        "dask": "dask.array",
        "xarray": "xarray",
    }.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    
    if failures:
        raise ImportError(
            "Dependency verification failed:\n"
            + "\n".join(f"{name}: {error}" for name, error in failures.items())
        )

import zarr, geff, tracksdata
print("All offline dependencies imported successfully.")

## Configuration

In [ ]:
# Weights. Three models now run in series, so three checkpoint sets. Point each
# at whatever dataset holds it; the defaults assume one dataset per model.
TRACK_W = "/kaggle/input/datasets/theoviel/biohub-cell-tracking-weights/"
CONTRASTIVE_W = "/kaggle/input/datasets/theoviel/biohub-contrastive-weights/"
EDGE_W = "/kaggle/input/datasets/theoviel/biohub-edge-weights/"

# The settings that produced cv 0.8506 out of fold on logs/edge_run/2026-09-09/0.
# Detection: the edge run's nodes came from preds_r6_0.40, i.e. NMS radius 6 um
# at threshold 0.40 -- *not* the 7.0 in the tracking run's args.json, which is
# what this stage would otherwise inherit.
DET_THRESHOLD = 0.40
NMS_RADIUS_UM = 6.0
# Edge threshold naming the output folder only: main_edge_postprocess re-solves
# the assignment from the candidate array at its own `threshold` (0.40), and
# the candidates are threshold-free, so this choice does not reach the result.
EDGE_THRESHOLD = 0.45

DATA_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development/test/"

import json as _json, os as _os
from pathlib import Path as _Path

# Kaggle allows 15360 MiB a GPU. Every stage fits in ~3 GB of *live* tensors,
# but PyTorch's default caching allocator hands back nothing and fragments
# badly on the edge model's variable-sized batches: measured on the largest
# stack (98 769 nodes) it reserved 48 980 MiB where 3023 MiB was allocated.
# That still completes on a 15 GB card -- it recycles once it hits the ceiling
# -- but it sits at 99.7% of the limit with the CUDA context on top, which is
# no margin at all. expandable_segments takes the same run to 3220 MiB
# reserved. Set here so the `!` subprocesses below inherit it.
_os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Which embeddings the edge checkpoint expects. The contrastive stage writes its
# vectors under a key naming its own run folder, and on Kaggle that folder is
# not the one it trained in -- so the key is taken from the edge run and forced,
# rather than left to be rederived from a path that has changed. A mismatch is
# not silent (load_video raises and lists what it found), but it is avoidable.
_edge_args = _json.load(open(_Path(EDGE_W) / "args.json"))
EMB_KEY = _edge_args.get("embeddings_key") or "embeddings"
print("edge run expects embeddings key:", EMB_KEY)
print("max_nodes_per_frame            :", _edge_args.get("max_nodes_per_frame"))


## 1. Detection and tracking

The U-Net detector plus the node transformer, as before. Two things differ from
the previous submission: the checkpoint carries the extra decoder stage (the
heatmap comes out at 1/2 Y/X while the node features stay on the isotropic 1/4
grid), and the NMS radius is passed explicitly rather than inherited from
`args.json` — see the note in the configuration cell.

Output is the node set every later stage reads.

In [ ]:
%%writefile inference.py

import sys
sys.path.append('./src/')

import warnings
warnings.filterwarnings("ignore")

import os
import json
import torch
import argparse
import numpy as np
from pathlib import Path

from tracking.main import build_model
from tracking_cellmot.io import list_datasets
from tracking.predict import PredictConfig, predict_video_thresholds
from training.distributed import cleanup_distributed, init_distributed
from utils.logger import Config

DATA_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development/test/"


def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--log_folder", required=True, help="run folder holding args.json and the checkpoint")
    p.add_argument("--ckpt", default="fold_fullfit/ckpt_final.pt",
                   help="checkpoint, relative to --log_folder; searched for if not found there")
    p.add_argument("--det_threshold", type=float, default=0.40,
                   help="detection threshold; pick it from the run's cv_score")
    p.add_argument("--nms_radius_um", type=float, default=None,
                   help="greedy-NMS radius. Overrides the training run's own "
                        "value, which is a training default and not the one the "
                        "downstream node set was chosen at -- preds_r6_0.40 "
                        "means radius 6, while args.json records 7")
    p.add_argument("--out_dir", default=None, help="default <log_folder>/preds")
    p.add_argument("--limit", type=int, default=None, help="cap the stacks (debugging)")
    p.add_argument("--data_dir", default=DATA_DIR, help="override the test split")
    return p.parse_args()


def resolve_ckpt(log_folder: Path, ckpt: str) -> Path:
    """The checkpoint, wherever the weights dataset happens to have put it.

    A run folder keeps checkpoints under fold_*/; a weights dataset is often
    flattened to the root instead. Rather than making the caller know which,
    take the path as given and fall back to a search by file name.
    """
    direct = log_folder / ckpt
    if direct.exists():
        return direct
    found = sorted(log_folder.rglob(Path(ckpt).name))
    if not found:
        available = sorted(str(f.relative_to(log_folder)) for f in log_folder.rglob("*.pt"))
        raise FileNotFoundError(
            f"{ckpt} not found under {log_folder}. Checkpoints present: {available}"
        )
    return found[0]


def main():
    args = parse_args()
    ctx = init_distributed(None)
    device = ctx.device

    paths = sorted(list_datasets(Path(args.data_dir), require_geff=False))
    if args.limit:
        paths = paths[: args.limit]

    log_folder = Path(args.log_folder)
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = resolve_ckpt(log_folder, args.ckpt)

    cfg = Config(json.load(open(log_folder / "args.json")))

    # Inference settings come from the training run wherever the run fixes
    # them, so the node sets the edge head sees here are the ones it was fitted
    # against. Only the detection threshold is chosen separately -- it is swept
    # after training, and the value below is the sweep's answer.
    pred_cfg = PredictConfig
    pred_cfg.det_threshold = args.det_threshold
    pred_cfg.normalization = getattr(cfg, "normalization", None)
    pred_cfg.nms = getattr(cfg, "nms", pred_cfg.nms)
    pred_cfg.nms_radius_um = (
        args.nms_radius_um if args.nms_radius_um is not None
        else getattr(cfg, "nms_radius_um", pred_cfg.nms_radius_um)
    )
    pred_cfg.pool_kernel_um = getattr(cfg, "pool_kernel_um", pred_cfg.pool_kernel_um)
    pred_cfg.use_offset = bool(getattr(cfg, "offset_head", False))

    window_size = getattr(cfg, "window_size", 2)
    downscale_factor = getattr(cfg, "downscale_factor", 1)

    if ctx.is_main:
        print(f"-> log_folder     : {log_folder}")
        print(f"-> checkpoint     : {ckpt_path.relative_to(log_folder)}")
        print(f"-> threshold      : {pred_cfg.det_threshold}")
        print(f"-> nms            : {pred_cfg.nms} @ {pred_cfg.nms_radius_um} um")
        print(f"-> normalization  : {pred_cfg.normalization}")
        print(f"-> window_size    : {window_size}")
        print(f"-> stacks         : {len(paths)}")
        print(f"-> writing to     : {out_dir}")
        if ctx.distributed:
            print(f"-> distributed    : {ctx.world_size} processes")
        print()

    state = torch.load(ckpt_path, map_location="cpu")
    state = state.get("model_state_dict", state)
    state = {k.replace("unet.module.", "unet.", 1): v for k, v in state.items()}

    model = build_model(cfg, None)
    model.load_state_dict(state, strict=True)
    model.eval().to(device)

    # Shard stacks across ranks; each writes its own npz files.
    mine = paths[ctx.rank :: ctx.world_size]

    for i, path in enumerate(mine):
        print(f"[rank {ctx.rank}] {i + 1}/{len(mine)}  -  {path.name}", flush=True)

        # predict_video_thresholds rather than predict_video: the latter drops
        # the per-node detection probability, and the edge model reads it as a
        # feature (`use_conf_score`). Without a `node_probs` array its dataset
        # refuses to build, so the pipeline would fail two stages later.
        results, _ = predict_video_thresholds(
            model,
            path,
            thresholds=[pred_cfg.det_threshold],
            device=device,
            cfg=pred_cfg,
            window_size=window_size,
            downscale_factor=downscale_factor,
            progress=ctx.is_main,
            return_det_logits=False,
            nms_radii=[pred_cfg.nms_radius_um],
        )
        coords, edges, node_probs = results[
            (float(pred_cfg.nms_radius_um), float(pred_cfg.det_threshold))
        ]

        np.savez_compressed(
            out_dir / f"{path.name}.npz",
            coords=coords,
            edges=np.asarray(edges, dtype=np.float64).reshape(-1, 4),
            # Row-aligned with coords, exactly as main_predict.py writes it.
            node_probs=np.asarray(node_probs, dtype=np.float32).reshape(-1),
        )


if __name__ == "__main__":
    try:
        main()
    finally:
        cleanup_distributed()


In [ ]:
# Detection settings are forced rather than inherited: NMS_RADIUS_UM is the
# radius the downstream node set was chosen at, which is not the one in the
# tracking run's args.json.
cmd = (
    "torchrun --standalone --nproc_per_node=2 inference.py"
    f" --log_folder '{TRACK_W}'"
    " --ckpt 'fold_fullfit/ckpt_final.pt'"
    f" --det_threshold {DET_THRESHOLD}"
    f" --nms_radius_um {NMS_RADIUS_UM}"
    " --out_dir /tmp/preds/"
)
print(cmd)
!{cmd}


## 2. Contrastive embeddings (NETr)

One appearance vector per detection, written **back into the detection npz**
rather than into a folder of its own, so the coordinates and their embeddings
cannot drift apart. The edge model reads them as node features; without them it
falls back to geometry alone and scores materially worse.

In [ ]:
# NETr embeddings for every detection, written back into /tmp/preds/*.npz under
# EMB_KEY. --fold fullfit runs the one model over every stack: the out-of-fold
# default has no meaning on a test split whose embryos are not the training ones.
cmd = (
    "torchrun --standalone --nproc_per_node=2 src/inference_contrastive.py"
    f" --exp_folder '{CONTRASTIVE_W}'"
    " --coords_log_folder /tmp/preds/"
    f" --data_dir '{DATA_DIR}'"
    " --fold fullfit"
    f" --key '{EMB_KEY}'"
)
print(cmd)
!{cmd}


## 3. Edge transformer

Scores every candidate link, and saves **all** of them alongside the committed
ones. That is what makes the threshold and the linking rule a post-processing
decision instead of a GPU one — the next cell re-solves the assignment from this
array, so the threshold here does not reach the final graph.

In [ ]:
%%writefile edge_inference.py
"""Score every candidate link with the edge transformer, on the test split.

`main_edge_predict.py` cannot be used as-is here: it picks its videos with
`load_videos(..., require_geff=True)` and slices them by fold and embryo, and
the test split has neither ground truth nor known embryo names. Everything below
the video list is the same library code that produced the out-of-fold numbers --
`load_video`, `dataset_for`, `score_video`, `link_video` -- so the only thing
this file changes is which stacks are fed in and which checkpoint reads them.

Writes coords + committed edges + the full scored candidate array, which is what
`main_edge_postprocess.py` re-links from.
"""

import sys
sys.path.append('./src/')

import warnings
warnings.filterwarnings("ignore")

import argparse
import json
from pathlib import Path

import dataclasses

import numpy as np
import torch

from edge_transformer.data import load_video
from edge_transformer.infer import dataset_for, load_hoct
from edge_transformer import predict as edge_predict
from edge_transformer.predict import link_video, score_video
from embeddings.dataset import find_coord_files
from tracking_cellmot.io import list_datasets
from training.distributed import cleanup_distributed, init_distributed


def parse_args():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--log_folder", required=True, help="edge run: args.json + checkpoint")
    p.add_argument("--ckpt", default="fold_fullfit/ckpt_final.pt")
    p.add_argument("--nodes_folder", required=True, help="detections, with embeddings")
    p.add_argument("--data_dir", required=True)
    p.add_argument("--out_dir", required=True)
    p.add_argument("--threshold", type=float, default=0.45)
    p.add_argument("--embeddings_key", default=None)
    p.add_argument("--max_nodes_per_frame", type=int, default=None,
                   help="EdgeDataset's own cap, which drops nodes at RANDOM and "
                        "re-draws per window. A training-memory knob; use "
                        "--max_nodes instead")
    p.add_argument("--edge_pair_budget", type=int, default=0,
                   help="cap on B * E_max^2 per forward, 0 = the library default "
                        "(32M). This, not --max_nodes, is what actually governs "
                        "peak GPU memory here -- halving it roughly halves the "
                        "peak, at some wall-clock cost")
    p.add_argument("--max_nodes", type=int, default=0,
                   help="keep only the N most confident detections per frame, "
                        "0 = no cap. A safety net against OOM on an unusually "
                        "crowded stack: the edge model's cost is quadratic in "
                        "the nodes of a frame pair")
    p.add_argument("--no_fp16", action="store_true")
    p.add_argument("--limit", type=int, default=None)
    return p.parse_args()


def resolve_ckpt(log_folder: Path, ckpt: str) -> Path:
    """The checkpoint wherever the weights dataset put it -- see inference.py."""
    direct = log_folder / ckpt
    if direct.exists():
        return direct
    found = sorted(log_folder.rglob(Path(ckpt).name))
    if not found:
        available = sorted(str(f.relative_to(log_folder)) for f in log_folder.rglob("*.pt"))
        raise FileNotFoundError(f"{ckpt} not found under {log_folder}. Present: {available}")
    return found[0]


def cap_nodes(video, max_nodes: int):
    """Keep the ``max_nodes`` most confident detections of each frame.

    A bound on the edge model's cost, which is quadratic in the nodes of a frame
    pair. Two things make this different from ``EdgeDataset``'s own
    ``max_nodes_per_frame``, and both matter for inference:

    * **It ranks by confidence.** The dataset's cap drops at random, which is
      right for training -- the survivors stay a fair sample -- and wrong here,
      where a dropped node gets no candidate edges at all and simply cannot be
      linked. If something has to go, it should be the detections the model was
      least sure of.
    * **It is applied once, to the node set.** The dataset's cap is re-drawn per
      window, so a node can be present in one window and absent from the next,
      and the graph a video produces is not a graph of any one node set.

    A cap that never binds costs nothing, so the default is off: the reported
    0.8506 is an uncapped number, and turning this on is a decision to trade
    recall for a memory ceiling.
    """
    if not max_nodes:
        return video
    if video.conf is None:
        raise ValueError(
            "--max_nodes ranks by detection confidence, but this node set has "
            "no 'node_probs' array. Re-run the detection stage, or drop the cap."
        )

    keep_parts = []
    for t in sorted(video.rows_by_frame):
        rows = video.rows_by_frame[t]
        if len(rows) <= max_nodes:
            keep_parts.append(rows)
            continue
        # stable, so ties fall back to file order rather than to whatever the
        # sort happens to do -- and keep ascending, since every consumer reads a
        # frame's rows in file order.
        best = np.argsort(-video.conf[rows], kind="stable")[:max_nodes]
        keep_parts.append(np.sort(rows[best]))
    keep = np.concatenate(keep_parts) if keep_parts else np.empty(0, int)

    if len(keep) == len(video.coords):
        return video

    coords = video.coords[keep]
    frame_of = coords[:, 0].astype(int)
    return dataclasses.replace(
        video,
        coords=coords,
        emb=None if video.emb is None else video.emb[keep],
        conf=video.conf[keep],
        rows_by_frame={int(t): np.flatnonzero(frame_of == t) for t in np.unique(frame_of)},
        # Inference has no ground-truth links to remap; refuse rather than
        # silently renumber them into nonsense if that ever changes.
        links={} if not video.links else _unsupported_links(),
    )


def _unsupported_links():
    raise NotImplementedError(
        "cap_nodes drops node rows, which would invalidate the ground-truth "
        "link indices. It is an inference-only stage."
    )


def candidate_array(video, scored):
    """(src row, tgt row, prob, distance um, dt) -- a copy of main_edge_predict's.

    Kept identical on purpose: it is the post-processing chain's entire input,
    and the out-of-fold tuning was done against exactly this layout.
    """
    frame_of = np.asarray(video.coords)[:, 0].astype(int)
    if not scored:
        return np.empty((0, 5))
    return np.asarray(
        [(a, b, p, d, frame_of[b] - frame_of[a]) for (a, b), (p, d) in scored.items()],
        dtype=np.float64,
    ).reshape(-1, 5)


def main():
    args = parse_args()
    ctx = init_distributed(None)
    device = ctx.device

    if args.edge_pair_budget:
        # A module constant read inside score_video's batch planner, so it is set
        # rather than passed. Done before any scoring, and echoed below, because
        # a memory knob that is not in the log is a memory knob nobody can
        # reproduce.
        edge_predict.EDGE_PAIR_BUDGET = int(args.edge_pair_budget)

    log_folder = Path(args.log_folder)
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = resolve_ckpt(log_folder, args.ckpt)

    model, cfg = load_hoct(str(ckpt_path), device=device, verbose=ctx.is_main)
    emb_key = args.embeddings_key or cfg.get("embeddings_key") or "embeddings"
    use_emb = bool(cfg.get("use_embeddings", True))

    # require_geff=False: the test split has no annotation. load_video already
    # copes -- it attaches links only when a .geff sits beside the stack -- so
    # the videos come back with an empty `links`, which is all inference needs.
    node_files = {p.name: p for p in find_coord_files(args.nodes_folder)}
    paths = sorted(list_datasets(Path(args.data_dir), require_geff=False))
    if args.limit:
        paths = paths[: args.limit]

    videos = []
    for path in paths:
        key = path.name + ".npz" if (path.name + ".npz") in node_files else path.name
        node_path = node_files.get(key)
        if node_path is None:
            if ctx.is_main:
                print(f"!  no detections for {path.name}, skipped")
            continue
        video = load_video(
            path.name, node_path, path,
            use_embeddings=use_emb, embeddings_key=emb_key,
        )
        n_before = len(video.coords)
        video = cap_nodes(video, args.max_nodes)
        if len(video.coords) != n_before and ctx.is_main:
            print(f"   {path.name}: {n_before} -> {len(video.coords)} nodes "
                  f"(cap {args.max_nodes}/frame)")
        videos.append(video)
    if not videos:
        raise FileNotFoundError(
            f"no stack of {args.data_dir} had a matching npz in {args.nodes_folder}"
        )

    dataset = dataset_for(cfg, videos, max_nodes_per_frame=args.max_nodes_per_frame)

    if ctx.is_main:
        print(f"-> checkpoint     : {ckpt_path}")
        print(f"-> nodes_folder   : {args.nodes_folder}")
        print(f"-> embeddings key : {emb_key}  (use_embeddings={use_emb})")
        print(f"-> videos         : {len(videos)}")
        total = sum(len(v.coords) for v in videos)
        per_frame = max((len(r) for v in videos for r in v.rows_by_frame.values()),
                        default=0)
        print(f"-> confidence cap : {args.max_nodes or 'none'} nodes/frame"
              f"   (widest frame after it: {per_frame}, {total} nodes total)")
        print(f"-> dataset cap    : {dataset.max_nodes_per_frame or 'none'}"
              f"   window {dataset.window_size}")
        print(f"-> edge budget    : {edge_predict.EDGE_PAIR_BUDGET:,} pairs/forward")
        print(f"-> threshold      : {args.threshold}")
        print(f"-> writing to     : {out_dir}")
        if ctx.distributed:
            print(f"-> distributed    : {ctx.world_size} processes")
        print()

    mine = list(range(len(videos)))[ctx.rank :: ctx.world_size]
    for i, vi in enumerate(mine):
        video = videos[vi]
        scored = score_video(model, dataset, vi, device, fp16=not args.no_fp16)
        coords, edges = link_video(video, scored, threshold=args.threshold)
        np.savez_compressed(
            out_dir / f"{video.name}.npz",
            coords=np.round(np.asarray(coords, np.float64)).astype(np.int16),
            edges=np.asarray(edges, np.float64).reshape(-1, 4),
            candidates=candidate_array(video, scored),
        )
        print(f"[rank {ctx.rank}] [{i + 1:>3}/{len(mine)}] {video.name}  "
              f"{len(coords)} nodes  {len(edges)} edges", flush=True)


if __name__ == "__main__":
    try:
        main()
    finally:
        cleanup_distributed()


In [ ]:
cmd = (
    "torchrun --standalone --nproc_per_node=2 edge_inference.py"
    f" --log_folder '{EDGE_W}'"
    " --ckpt 'fold_fullfit/ckpt_final.pt'"
    " --nodes_folder /tmp/preds/"
    f" --data_dir '{DATA_DIR}'"
    " --out_dir /tmp/preds_edge/"
    f" --threshold {EDGE_THRESHOLD}"
    f" --embeddings_key '{EMB_KEY}'"
)
print(cmd)
!{cmd}


## 4. Edge post-processing

`src/main_edge_postprocess.py` on its tuned defaults, which are the ones that
scored **cv 0.8506** over the 199 out-of-fold stacks of
`logs/edge_run/2026-09-09/0/preds_0.45`:

    assign(th=0.4) -> recover(gap<=3, r=4) -> divisions(p>=0.4, diverge=3.5)
        -> prune(min_track=7) -> linefit(w=0.6)

The ILP node filter is off. It is worth +0.0002 once the divisions are refit to
its graph, and costs 9-16 s a stack against ~1 s for the rest of the chain — the
tightened `recover` and `min_track` now do the work it used to.

`--no_score` because the test split ships no ground truth.

In [ ]:
!python src/main_edge_postprocess.py \
    --preds_dir "/tmp/preds_edge/" \
    --out_dir "/tmp/preds_pp/" \
    --geff_dir "/tmp/preds_pp_geff/" \
    --no_score \
    --jobs 4


## Submission

`src/submission.py` again, rather than a copy — the two had already converged
to the same file apart from a warnings filter.

In [ ]:
!python src/submission.py /tmp/preds_pp_geff -o ./submission.csv

In [ ]:
sub = pd.read_csv('./submission.csv')
sub.sample(5)